# Backtest - 2025/26

Every gameweek of a completed season, projected as if from the deadline, then
compared against what actually happened.

One question is asked of each model separately: does it predict the right
amount of the thing it's meant to predict? The goals model should predict about
as many goals as were scored, the clean sheet model about as many clean sheets,
and so on.

Everything is measured in natural units (goals, clean sheets, hits, minutes)
rather than points, because a component can be right about points for the wrong
reason, and natural units are what make an error interpretable.

Runtime is about a minute.

### Where the odds come from

football-data.co.uk publishes actual Betfair Exchange prices: match odds and
over/under 2.5, exactly the pair the scoreline model fits. The **pre-close**
columns are used, not the closing ones, because closing prices are taken at
kickoff, which for a Sunday fixture is a day or two after the FPL deadline and
carries team news no manager could have had.

In [1]:
import sys, time
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "fplfh").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

try:                                   # pick up code edits without a restart
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except Exception:
    pass

import pandas as pd

from fplfh.evaluate import (
    evaluate_season, load_history, build_historical_fixtures, audit_look_ahead,
    score_components, score_scorelines, score_distribution,
    goal_count_distribution,
)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

SEASON = "2025-26"

## 1. The look-ahead check

Before trusting anything below, confirm the reconstruction cannot see the
gameweeks it is predicting.

The obvious test - no player may hold more than 90 × (GW − 1) minutes - is
**wrong**, and quietly so. Double gameweeks mean a team can play more matches
than gameweeks, so before GW30 Arsenal had played 30 matches in 29 gameweeks and
their keeper held 2,700 minutes against a supposed ceiling of 2,610. That test
raises a false alarm rather than catching anything, and the
`naive_ceiling_would_flag` column below shows it doing exactly that.

Two sound tests replace it:

1. **Exact reconstruction** - every cumulative stat must equal the sum over
   gameweeks strictly before the target, to floating-point tolerance. This is
   airtight: had any future row contributed, the totals would not match.
2. **A fixture-aware ceiling** - minutes cannot exceed 90 × the matches the
   player's team actually played beforehand.

In [2]:
history = load_history(SEASON)
fixtures, id_to_name = build_historical_fixtures(history)

audit = audit_look_ahead(history, fixtures, id_to_name)
print(audit.to_string(index=False))
assert audit["passes"].all(), "look-ahead detected"
print("\nall gameweeks pass")

 gameweek  players  max_abs_stat_error  players_over_fixture_ceiling  naive_ceiling_would_flag  passes
        4      712               0.000                             0                         0    True
       10      746               0.000                             0                         0    True
       20      780               0.000                             0                         0    True
       30      820               0.000                             0                         1    True
       38      840               0.000                             0                         0    True

all gameweeks pass


## 2. Run the season

Gameweeks 1 and 2 are skipped: with one game of history every rate is still
essentially its prior, so those rows would measure the prior rather than the
model.

Scorelines come from the market where a fixture was priced and from the ratings
model where it was not - exactly what the live tool does, so these numbers
describe the tool as it is actually used.

In [3]:
t0 = time.time()
per_player, per_fixture = evaluate_season(SEASON)
print(f"\ndone in {time.time() - t0:.0f}s")
print(f"  player-gameweek projections : {len(per_player):,}")
print(f"  gameweeks covered           : {per_player.event.min()}-{per_player.event.max()}")

2025-26: 380 fixtures, 360 priced by the market


  gameweek 5


  gameweek 10


  gameweek 15


  gameweek 20


  gameweek 25


  gameweek 30


  gameweek 35



done in 27s
  player-gameweek projections : 27,807
  gameweeks covered           : 3-38


## 3. Each model against what happened

`predicted` is what the model expected across the whole season; `actual` is what
occurred. `pct_error` is the column to read - a model is doing its job when it
lands within a few percent. Anything in double figures is a real defect rather
than noise, given the sample size here.

In [4]:
print(score_components(per_player).sort_values("pct_error", key=abs).to_string(index=False))

   component   predicted      actual       diff  pct_error  per_player_pred  per_player_actual
yellow cards   1,358.829   1,354.000      4.829      0.357            0.049              0.049
clean sheets   1,860.953   1,848.000     12.953      0.701            0.067              0.066
       saves   2,003.779   2,021.000    -17.221     -0.852            0.072              0.073
     minutes 700,746.685 707,943.000 -7,196.315     -1.017           25.200             25.459
bonus points   2,318.734   2,294.000     24.734      1.078            0.083              0.082
 appearances  11,038.877  10,850.000    188.877      1.741            0.397              0.390
       goals   1,008.478     957.000     51.478      5.379            0.036              0.034
 defcon hits   1,275.985   1,390.000   -114.015     -8.203            0.046              0.050
     assists     965.657     892.000     73.657      8.257            0.035              0.032


## 4. The scoreline model: does it get the average right?

The same question one level up, before any player is involved. This is the model
the market prices feed directly, so it is the cleanest read on whether they are
doing their job.

In [5]:
print(score_scorelines(per_fixture).to_string(index=False))

    quantity  per_match_pred  per_match_actual  diff  pct_error
  home goals           1.595             1.525 0.070      4.596
  away goals           1.278             1.233 0.045      3.627
 total goals           2.873             2.758 0.115      4.163
clean sheets           0.262             0.249 0.013      5.358


## 5. The scoreline model: does it get the *shape* right?

Matching the average is a weak test. A model can predict exactly the right number
of goals per match while getting the distribution wrong - too many 1-1s, not
enough 0-0s and 4-2s - and that matters a great deal here, because every clean
sheet, and therefore every defender and goalkeeper, is priced off the shape
rather than off the mean.

This is also the specific thing the model is built to get right. It is a
**Dixon-Coles** bivariate Poisson, and the entire purpose of its low-score
correction τ is that independent Poissons understate 0-0 and 1-1 while
overstating 1-0 and 0-1. If τ is doing its job, those four scores should come out
close.

In [6]:
dist = score_distribution(per_fixture, max_goals=4)
print("predicted vs observed frequency of each correct score")
print(dist.to_string(index=False))

predicted vs observed frequency of each correct score
         score  predicted  observed  pred_pct  obs_pct  diff_pct
           1-1     41.400    44.000    11.500   12.222    -0.722
           2-1     29.682    37.000     8.245   10.278    -2.033
           0-1     23.839    30.000     6.622    8.333    -1.711
           1-2     24.770    27.000     6.881    7.500    -0.619
           2-0     26.634    27.000     7.398    7.500    -0.102
           2-2     18.003    26.000     5.001    7.222    -2.221
           3-0     15.681    25.000     4.356    6.944    -2.589
           0-0     24.199    25.000     6.722    6.944    -0.223
           1-0     29.372    19.000     8.159    5.278     2.881
           3-1     16.451    19.000     4.570    5.278    -0.708
           0-2     18.425    14.000     5.118    3.889     1.229
           3-2      9.398    11.000     2.610    3.056    -0.445
           0-3      9.091    10.000     2.525    2.778    -0.253
           1-3     11.520     9.000 

In [7]:
# The four scores the Dixon-Coles correction exists to fix.
print("The low-score corner, which tau is there to get right:")
print(dist[dist.score.isin(["0-0", "1-1", "1-0", "0-1"])]
      [["score", "pred_pct", "obs_pct", "diff_pct"]].to_string(index=False))
print()
print(f"  mean |error| over all correct scores : {dist.diff_pct.abs().mean():.2f} pp")
print(f"  largest single miss                  : {dist.diff_pct.abs().max():.2f} pp"
      f"  ({dist.loc[dist.diff_pct.abs().idxmax(), 'score']})")

The low-score corner, which tau is there to get right:
score  pred_pct  obs_pct  diff_pct
  1-1    11.500   12.222    -0.722
  0-1     6.622    8.333    -1.711
  0-0     6.722    6.944    -0.223
  1-0     8.159    5.278     2.881

  mean |error| over all correct scores : 0.95 pp
  largest single miss                  : 3.62 pp  (>4 either side)


The one-dimensional view of the same question - total goals in a match. This is
what the over/under market prices directly, so a mismatch here points at the
market input rather than at the correlation structure between the two teams.

In [8]:
print(goal_count_distribution(per_fixture).to_string(index=False))

total_goals  predicted  observed  pred_pct  obs_pct  diff_pct
          0     24.199    25.000     6.722    6.944    -0.223
          1     53.211    49.000    14.781   13.611     1.170
          2     86.459    85.000    24.016   23.611     0.405
          3     79.224    99.000    22.007   27.500    -5.493
          4     56.968    57.000    15.824   15.833    -0.009
          5     33.103    31.000     9.195    8.611     0.584
          6     16.192     7.000     4.498    1.944     2.553
         7+     10.644     7.000     2.957    1.944     1.012


## 6. Is the error spread evenly, or concentrated?

A component that is 10% out overall might be 10% out everywhere, or fine for
three positions and badly wrong for one. The two call for different fixes, so it
is worth a look before concluding anything from section 3.

In [9]:
print(score_components(per_player, by="position")
      .pivot(index="component", columns="position", values="pct_error")
      .round(1).to_string())

position         DEF     FWD     GKP    MID
component                                  
appearances    9.700 -14.900  33.200 -4.200
assists       10.700 -35.300 -54.100 15.600
bonus points -20.800   5.100 -41.600 18.500
clean sheets   3.000     NaN   8.100 -2.700
defcon hits  -11.000 -16.300     NaN -4.200
goals         17.200   2.200     NaN  4.300
minutes       -1.900  -1.500  -2.500  0.200
saves            NaN     NaN  -0.900    NaN
yellow cards   4.500 -12.000   0.100 -0.800


In [10]:
# The same split by price, which separates nailed starters from fringe players
# more cleanly than position does.
d = per_player.copy()
d["price_band"] = pd.cut(d["price"], [0, 4.5, 5.5, 7.0, 9.0, 30.0],
                         labels=["<=4.5", "4.6-5.5", "5.6-7.0", "7.1-9.0", "9.0+"])
print(score_components(d, by="price_band")
      .pivot(index="component", columns="price_band", values="pct_error")
      .round(1).to_string())

price_band    4.6-5.5  5.6-7.0  7.1-9.0    9.0+   <=4.5
component                                              
appearances    -7.000  -10.500   -8.100  -1.600  30.700
assists         2.600   19.000    6.200 -26.300  17.200
bonus points   -5.200   15.600    5.600   9.400 -14.600
clean sheets   -4.100   -3.800  -12.700   0.700  18.900
defcon hits   -11.500    5.200  -16.900 -24.500  -7.000
goals          -7.100   10.400   -0.600  26.900  42.500
minutes        -4.000   -5.100   -7.100  -8.800   9.500
saves          -7.900    0.100      NaN     NaN   7.700
yellow cards   -8.600    0.700    1.800  15.100  16.000


## 7. Where look-ahead could still get in

Section 1 proves the player reconstruction is clean. That is not the whole
surface. Every place the backtest could learn something it should not, and what
was done about it:

| stage | risk | status |
|---|---|---|
| player stats | season-to-date totals include the target gameweek | **closed** - rebuilt from earlier gameweeks only, asserted in section 1 |
| minutes history | same | **closed** - `estimate_minutes` filters to `event < target` internally, so a careless caller cannot leak |
| fixture results | the fixture frame carries final scores | **closed** - `build_player_fixtures` selects an explicit column list; `home_goals`/`away_goals` never reach the model |
| team ratings | `team_ratings` takes the fixture list | **closed** - the argument is unused; ratings are identical when passed an empty frame |
| odds | closing prices are taken at kickoff, after the deadline | **closed** - pre-close columns used. Costs accuracy: mean drift in P(home) to close is 0.021, correlation 0.989 |
| prices | a gameweek's `value` is read from its own row | **acceptable** - FPL prices move overnight, so that figure stands at the deadline. 7.4% of rows move week to week, mean 0.1m. Affects affordability, not expected points |
| availability | injury flags are not in the archive | **closed, and costly** - every player is treated as fit, a handicap the live tool does not carry |
| **parameters** | **every constant was fitted on 2025/26** | **OPEN - the real one** |

### The one that is genuinely open

Twelve constants in `config/model_params.yaml` - DefCon dispersion, the saves
intercept and slope, assists-per-goal, the bench curve, substitutes per team,
P(60 | start), the opponent-scaling beta, the appearance-mode minutes - were all
estimated on 2025/26, and this notebook then scores them on 2025/26.

That is not look-ahead in the walk-forward sense: no gameweek sees its own
result. It is **in-sample fitting**, and it flatters every number here by an
amount this notebook cannot measure. Two components make the exposure concrete:
the bench curve was fitted on 21,395 non-start observations *from this season*,
and DefCon dispersion was solved so the modelled hit-rate reproduces *this
season's* observed rate.

The honest reading is that these figures are an **upper bound** on live accuracy.
The only way to resolve it is out-of-sample: either refit on 2025/26 and score on
2026/27, or hold out a block of gameweeks and refit without them. Neither is done
here.

### A second limit, unrelated to look-ahead

Predicting the right total is necessary, not sufficient. A model could predict
exactly the right number of goals league-wide while attributing them to entirely
the wrong players. Section 6 is a partial check on that, and it already finds
components that pass on the total and fail on the split.